# 🧩 实验 10：在 CartPole 上实现 REINFORCE

在之前的实验中，我们使用蒙特卡洛方法来估计价值函数，从而解决 CartPole 控制任务。我们将状态空间离散化，收集完整轨迹，计算回报，并使用这些回报来更新表格形式的 \(Q(s,a)\) 估计值。

在本实验中，我们将重新回到 CartPole 环境。不过这一次，我们不再估计价值函数，而是直接使用神经网络来学习一个**参数化策略**。这种方法被称为**策略梯度**。

与基于 Q 表选择动作不同，策略网络会输出一个关于动作的概率分布。我们会更新网络参数，使得那些带来更高回报的动作在未来被选择的概率变大。

我们的目标是实现 **REINFORCE**。REINFORCE 是最简单的策略梯度算法之一，其基本流程如下：

- 使用当前策略收集完整 episode；
- 计算每个时间步的蒙特卡洛回报；
- 调整策略参数，使得高回报动作的对数概率增大。

In [ ]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.nn.functional as F

In [ ]:
env = gym.make("CartPole-v1")   # no need to discretize now
obs, info = env.reset(seed=0)
obs_dim = env.observation_space.shape[0]  # 4 for CartPole
n_actions = env.action_space.n            # 2 for CartPole

### 任务 1：定义策略网络 `PolicyNet`

在这一部分，你需要实现一个小型神经网络，用来表示 CartPole 中的策略：

\[
\pi_\theta(a \mid s)
\]

`CartPole-v1` 的观测空间是一个 4 维向量，包括：

- 小车位置；
- 小车速度；
- 杆子的角度；
- 杆子的角速度。

动作空间包含 2 个离散动作：

- `0`：将小车向左推；
- `1`：将小车向右推。

我们将使用一个**多层感知机**，也就是 **MLP**。这个网络需要：

- 输入观测状态 \(s \in \mathbb{R}^4\)；
- 输出 2 个动作对应的 **logits**，这些 logits 后续会传入 `Categorical` 分布；
- 使用两个隐藏层，并在隐藏层后使用 ReLU 激活函数。

网络结构提示：

- 使用 `nn.Sequential` 将多层网络堆叠起来。
- 对于 CartPole，一个常见的选择是：
  - 第一个隐藏层：大约 100–150 个神经元，例如 `128`；
  - 第二个隐藏层：可以更小一些，例如第一个隐藏层的一半左右，如 `64`。
- 最后一层线性层应该从第二个隐藏层映射到 `n_actions`。
- 输出层后不要加激活函数；`Categorical` 分布会在内部处理 softmax。

In [ ]:
# ----- Policy network π_θ(a | s) -----
class PolicyNet(nn.Module):
    def __init__(self, obs_dim, n_actions):
        super().__init__()
        # Your time to work on it
        self.net = ####

    def forward(self, x):
        logits = self.net(x)  # shape: (batch, n_actions)
        return torch.distributions.Categorical(logits=logits)

In [ ]:
policy = PolicyNet(obs_dim, n_actions)
optimizer = optim.Adam(policy.parameters(), lr=1e-4)
gamma = 0.99
num_episodes = 200000

### 任务 2：实现 REINFORCE 损失函数

在收集完一个完整的 episode，并计算出每个时间步对应的蒙特卡洛回报 \(G_t\) 之后，最后一步就是更新策略网络的参数。

在 REINFORCE 中，我们希望沿着能够增加高回报动作对数概率的方向来调整策略。也就是说，如果某个动作最终带来了较高的回报，那么之后在相似状态下，策略应该更倾向于选择这个动作。

回忆一下 REINFORCE 的参数更新公式：

\[
\theta \leftarrow \theta + \alpha \, \nabla_\theta 
\log \pi_\theta(a_t \mid s_t) \, G_t.
\]

在实际实现中，我们通常不会手动写出这个参数更新公式。相反，我们会构造一个**损失函数**，使得对这个损失函数执行梯度下降时，效果等价于对目标函数 \(J(\theta)\) 执行梯度上升。

你的任务：

1. 你已经有一个 `log_probs` 列表，其中每一项对应 episode 中某一步所采取动作的 log-probability。
2. 你已经有一个 `returns` 列表，其中包含每个时间步对应的蒙特卡洛回报 \(G_t\)。
3. 请将它们组合成一个单一的标量损失函数。

In [ ]:
returns_history = []

for ep in range(1, num_episodes + 1):
    obs, _ = env.reset()
    done = False

    log_probs = []   # store log π_θ(a_t | s_t)
    rewards = []     # store r_t

    # Generate an episode
    while not done:
        s_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)   # shape (1, obs_dim)
        dist = policy(s_t)                                          # π_θ(. | s_t)
        a_t = dist.sample()                                         # sample action
        log_prob_t = dist.log_prob(a_t)                             # log π_θ(a_t | s_t)

        obs_next, r, term, trunc, _ = env.step(a_t.item())
        done = term or trunc

        log_probs.append(log_prob_t)
        rewards.append(r)

        obs = obs_next

    #  Value update
    T = len(rewards)
    G = 0.0
    returns = []

    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)

    returns = torch.tensor(returns, dtype=torch.float32)
    log_probs = torch.stack(log_probs)      # shape (T,)

    # Your time to work on it
    loss = ####

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    ep_return = sum(rewards)
    returns_history.append(ep_return)

    if ep % 100 == 0:
        avg = np.mean(returns_history[-50:])
        print(f"Episode {ep:4d} | Return: {ep_return:4.1f} | "
              f"Avg(50): {avg:5.1f}")

env.close()